# Lab 02 — Measure the Optimism Gap on CAMELS-US

**EN5425 / EV4240 · Week 2 · 60 minutes in class · note due Sunday 23:59 KST**

You will train two models to predict how much water leaves a river basin, tune one with a
hyperparameter sweep, and then ask the only question that matters: **does the score survive an
honest split?** You will measure the **optimism gap** — score(random split) − score(spatial split) —
on real data, yourself.

**How to work:** run the cells top to bottom. Cells marked ✍️ have a small piece for **you** to
complete or check — use your coding agent freely, but read what it writes.

| step | time | what happens |
|---|---|---|
| 1 | 10′ | load the basin table, choose features honestly |
| 2 | 10′ | two baselines under a **random** split |
| 3 | 10′ | Weights & Biases sweep → pick a winner |
| 4 | 15′ | re-evaluate with a **spatial** split → the gap |
| 5 | 10′ | write `notes/week02.md`, export PDF, submit |

*Data: CAMELS-US catchment attributes (Newman et al. 2015; Addor et al. 2017), 671 basins.
A course copy is loaded straight from the web in Step 1 — nothing to download by hand.*


In [ ]:
# Setup — one minute. (Colab already has torch, sklearn, pandas.)
%pip install -q wandb

import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.model_selection import KFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
print('torch', torch.__version__, '· gpu:', torch.cuda.is_available())


In [ ]:
# Weights & Biases — sign in once (free account: wandb.ai).
# If the login prompt gives you trouble today, the fallback line keeps everything
# working offline — but sweeps + the parallel-coordinates plot need a real login.
import wandb, os
try:
    wandb.login(timeout=120)
except Exception as e:
    print('login failed → running offline:', e)
    os.environ['WANDB_MODE'] = 'offline'


## Step 1 · Load the table, choose features honestly (10′)

One row per basin: 59 attributes + our two targets, **runoff_ratio** and **q_mean**
(mean daily discharge, mm/day — we model `log(q_mean)` because it spans orders of magnitude).

Two groups of columns are **banned as features**, and the *why* is this week's lecture:

1. **Every hydrological signature** (`baseflow_index`, `q5`, `q95`, `high_q_freq`, …) — they are
   *computed from the basin's own streamflow record*, i.e. from the target. Leakage before you even split
   (checklist item 4).
2. **`gauge_lat`, `gauge_lon`** — literally the basin's location. A model given coordinates can
   memorize geography, which is exactly what the spatial split is supposed to test.


In [ ]:
df = pd.read_csv(
    'https://raw.githubusercontent.com/Hyunglok-Kim/en5425-student-template/main/labs/data/camels_attributes.csv',
    dtype={'gauge_id': str, 'huc_02': str})
print(df.shape)

HYDRO_SIGNATURES = ['q_mean', 'runoff_ratio', 'slope_fdc', 'baseflow_index', 'stream_elas',
                    'q5', 'q95', 'high_q_freq', 'high_q_dur', 'low_q_freq', 'low_q_dur',
                    'zero_q_freq', 'hfd_mean']
COORDS = ['gauge_lat', 'gauge_lon']

numeric = df.select_dtypes(include=[np.number]).columns
FEATURES = [c for c in numeric if c not in HYDRO_SIGNATURES + COORDS]
print(len(FEATURES), 'features:', FEATURES)

data = df.dropna(subset=FEATURES + ['runoff_ratio', 'q_mean', 'huc_02']).reset_index(drop=True)
X = data[FEATURES].to_numpy(dtype=np.float32)
targets = {'runoff_ratio': data['runoff_ratio'].to_numpy(np.float32),
           'log_q_mean':   np.log(data['q_mean'].to_numpy(np.float32))}
groups = data['huc_02'].to_numpy()   # 2-digit hydrologic region — our spatial blocks
print(f'{len(data)} basins · {len(FEATURES)} features · {data.huc_02.nunique()} regions')

# ✍️ sanity check — run this and read it: no target-derived column may survive.
assert not set(FEATURES) & set(HYDRO_SIGNATURES + COORDS)


## Step 2 · Two baselines under a random split (10′)

- **Gradient boosting** (`HistGradientBoostingRegressor`) — the honest tabular baseline.
- A **small neural network** (2 hidden layers, PyTorch, AdamW — last week's recipe in miniature).

⚠️ **The scaler is fit inside each training fold** — fitting it on all 671 basins would leak the
test folds' statistics (checklist item 4 again). Look for it in `evaluate()` below.


In [ ]:
def make_mlp(n_in, width=64, depth=2, dropout=0.2):
    layers, d = [], n_in
    for _ in range(depth):
        layers += [nn.Linear(d, width), nn.ReLU(), nn.Dropout(dropout)]; d = width
    layers += [nn.Linear(d, 1)]
    return nn.Sequential(*layers)

def fit_mlp(Xtr, ytr, width=64, depth=2, dropout=0.2, lr=1e-3, epochs=200):
    net = make_mlp(Xtr.shape[1], width, depth, dropout)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-2)
    xb, yb = torch.tensor(Xtr), torch.tensor(ytr).unsqueeze(1)
    net.train()
    for _ in range(epochs):
        opt.zero_grad(); loss = nn.functional.mse_loss(net(xb), yb)
        loss.backward(); opt.step()
    return net

def evaluate(kind, y, splitter, groups=None, **mlp_kw):
    """kind: 'gbm' or 'mlp'. Returns per-fold R2 list."""
    scores = []
    for tr, te in splitter.split(X, y, groups):
        scaler = StandardScaler().fit(X[tr])          # <-- fold-internal: train rows only
        Xtr, Xte = scaler.transform(X[tr]), scaler.transform(X[te])
        if kind == 'gbm':
            model = HistGradientBoostingRegressor(random_state=SEED).fit(Xtr, y[tr])
            pred = model.predict(Xte)
        else:
            net = fit_mlp(Xtr.astype(np.float32), y[tr], **mlp_kw)
            net.eval()
            with torch.no_grad():
                pred = net(torch.tensor(Xte.astype(np.float32))).squeeze(1).numpy()
        scores.append(r2_score(y[te], pred))
    return np.array(scores)

random_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

for tname, y in targets.items():
    for kind in ('gbm', 'mlp'):
        s = evaluate(kind, y, random_cv)
        print(f'random split · {kind} · {tname}:  R² = {s.mean():.3f} ± {s.std():.3f}')


## Step 3 · Sweep the network with Weights & Biases (10′)

A small random sweep (12 runs) over learning rate, width, depth, dropout — selected on the
**random-split** score, exactly how most papers tune. When it finishes, open the sweep link,
add a **parallel-coordinates panel**, and export the image for your note.


In [ ]:
sweep_config = {
    'method': 'random',
    'metric': {'name': 'r2_random', 'goal': 'maximize'},
    'parameters': {
        'lr':      {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 1e-2},
        'width':   {'values': [32, 64, 128]},
        'depth':   {'values': [2, 3]},
        'dropout': {'values': [0.0, 0.2, 0.4]},
    },
}

y_sweep = targets['runoff_ratio']
results = []

def sweep_run():
    with wandb.init(project='en5425-lab02') as run:
        c = run.config
        s = evaluate('mlp', y_sweep, random_cv,
                     width=c.width, depth=c.depth, dropout=c.dropout, lr=c.lr)
        run.log({'r2_random': s.mean()})
        results.append({'lr': c.lr, 'width': c.width, 'depth': c.depth,
                        'dropout': c.dropout, 'r2_random': s.mean()})

sweep_id = wandb.sweep(sweep_config, project='en5425-lab02')
wandb.agent(sweep_id, function=sweep_run, count=12)

best = max(results, key=lambda r: r['r2_random'])
print('sweep winner:', best)


## Step 4 · The experiment: re-evaluate honestly (15′)

Same data, same winning model — one change: `GroupKFold` with **huc_02** as groups, so whole
hydrologic regions are held out and a fold never sees its neighbors.

✍️ *Before you run the next cell, write down your prediction: how much will each score drop?*


In [ ]:
spatial_cv = GroupKFold(n_splits=5)
mlp_kw = {k: best[k] for k in ('width', 'depth', 'dropout', 'lr')}

rows = []
for tname, y in targets.items():
    for kind, kw in (('gbm', {}), ('tuned mlp', mlp_kw)):
        r_rand = evaluate(kind.split()[-1] if kind != 'gbm' else 'gbm', y, random_cv, **kw)
        r_spat = evaluate(kind.split()[-1] if kind != 'gbm' else 'gbm', y, spatial_cv, groups=groups, **kw)
        rows.append({'model': kind, 'target': tname,
                     'R2 random': round(r_rand.mean(), 3),
                     'R2 spatial (huc_02)': round(r_spat.mean(), 3),
                     'optimism gap': round(r_rand.mean() - r_spat.mean(), 3)})

table = pd.DataFrame(rows)
print(table.to_string(index=False))
print()
print(table.to_markdown(index=False))   # ← paste this straight into notes/week02.md


## Step 5 · Write Note 02 (10′) — this is the deliverable

Copy `notes/TEMPLATE.md` → `notes/week02.md` in your semester repository and include:

1. the **parallel-coordinates plot** exported from your sweep page,
2. the **table** printed above (markdown version is ready to paste),
3. a **one-paragraph leakage post-mortem**: which checklist item does the random split violate,
   what did the gap measure, and what split would you defend in front of a reviewer?

Export the note to PDF and submit it in the **“My research note (Week 2)”** box on the Week 2
page of the course site — **Sunday 23:59 KST**.

*If your gap came out near zero — that is also a finding. Which features carry information that
genuinely transfers across regions? Say so in the post-mortem.*

*Stuck? Ask your agent to explain any cell — then make it show you where the scaler is fit and
check it is inside the fold. Still stuck: S6 building, room 317 — the HydroAI lab researchers
are expecting you.*
